# Build `spanish_subset_collapsed/train_bt.csv` and `test_bt.csv`

Run this notebook **once** after `translate.py` has produced the sentence-level `_bt.csv` files.

It collapses the sentence-level back-translations to comment level (joining DETESTS sentences
with a space, matching the collapse done in `explore_detests.ipynb`) and saves
`comment_id` + `backtranslated_text` to `data/spanish_subset_collapsed/train_bt.csv` (and `test_bt.csv`).

`data/loader.py` then merges these into the collapsed train/test CSVs on `comment_id`.

In [ ]:
import re
import os
import pandas as pd

RAW_DIR      = 'data/spanish_subset/'
COLLAPSE_DIR = 'data/spanish_subset_collapsed/'
os.makedirs(COLLAPSE_DIR, exist_ok=True)

In [ ]:
def collapse_bt(df, df_bt):
    """
    Positionally merge backtranslated_text (df and df_bt must have the same
    row order and length), then:
    - DETESTS: join sentences per consistent comment_id with a space
    - Stereohoax: keep the single bt entry per row as-is
    Returns a DataFrame with only [comment_id, backtranslated_text].
    """
    assert len(df) == len(df_bt), (
        f'Row count mismatch: {len(df)} (original) vs {len(df_bt)} (bt)'
    )

    df = df.copy()
    df['backtranslated_text'] = df_bt['backtranslated_text'].values

    detests = df[df['source'] == 'detests'].copy()
    other   = df[df['source'] != 'detests'][['comment_id', 'backtranslated_text']].copy()

    consistent_ids = (
        detests.groupby('comment_id')['stereotype']
        .nunique()
        .pipe(lambda s: s[s == 1].index)
    )
    print(f'  Consistent comment_ids: {len(consistent_ids):,} / {detests["comment_id"].nunique():,}')

    collapsed = (
        detests[detests['comment_id'].isin(consistent_ids)]
        .sort_values('id')
        .groupby('comment_id', as_index=False)
        .agg(backtranslated_text=('backtranslated_text', ' '.join))
    )

    return pd.concat([other, collapsed], ignore_index=True)

In [ ]:
for split in ['train', 'test']:
    bt_path = RAW_DIR + f'{split}_bt.csv'
    if not os.path.exists(bt_path):
        print(f'Skipping {split}: {bt_path} not found. Run translate.py first.')
        continue

    df    = pd.read_csv(RAW_DIR + f'{split}.csv')
    df_bt = pd.read_csv(bt_path)

    print(f'Processing {split}  ({len(df):,} rows)...')
    result = collapse_bt(df, df_bt)

    out_path = COLLAPSE_DIR + f'{split}_bt.csv'
    result.to_csv(out_path, index=False)
    print(f'  Saved {len(result):,} rows → {out_path}\n')

print('Done.')

In [ ]:
# Sanity check
for split in ['train', 'test']:
    out_path = COLLAPSE_DIR + f'{split}_bt.csv'
    if not os.path.exists(out_path):
        continue
    df = pd.read_csv(out_path)
    print(f'{split}_bt.csv — {len(df):,} rows, columns: {list(df.columns)}')
    print(df.head(3).to_string())
    print()